# Banking Fraud Intelligence
## Can transaction history improve the review queue?

**P3 · Model development and August validation · Simulated card transactions**

Compare two ways to fill a daily investigation queue: rank by transaction amount, or rank by a logistic regression model that also uses earlier customer and terminal activity. The same August transactions and daily review limits apply to both policies.

Open the notebook in VS Code, select your Python 3.12 kernel and choose **Run All**. Use the same local CSV as the [feature engineering notebook](02_fraud_feature_engineering.ipynb). Install the pinned scikit-learn version in that kernel the first time. [Setup](../data/README.md#run-the-modelling-notebook)

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

repo = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "fraud_model.py").is_file()),
    None,
)
if repo is None:
    raise FileNotFoundError("Open this notebook from the Banking Fraud Intelligence checkout.")
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from src.fraud_model import compare_review_queues, fit_ranker, score_transactions
from src.transaction_features import MODEL_FEATURES, SPLIT_PERIODS, build_features
from src.validate_transactions import load_transactions

## 1 · Fit on April through July

Logistic regression gives us a simple first model. It uses the seven P2 predictors, including the current amount. The pipeline learns median replacements and standardisation from April through July, adds missing-history indicators and weights the fraud and legitimate classes inversely to their training frequencies. The score is used for ranking.

The loader checks source integrity. Feature preparation for this notebook ends before September, and the model and preprocessing see April through July only. August supplies the validation results.

In [ ]:
transactions = load_transactions()
validation_start, validation_stop = SPLIT_PERIODS["validation"]
development = transactions.loc[transactions["TX_DATETIME"] < validation_stop]
features = build_features(development)
train = features.loc[features["TX_DATETIME"] < validation_start]
validation = features.loc[features["TX_DATETIME"] >= validation_start]

assert not train.empty and not validation.empty
assert train["TX_DATETIME"].max() < validation["TX_DATETIME"].min()
assert validation["TX_DATETIME"].max() < validation_stop
assert len(train) + len(validation) == len(development)
assert set(MODEL_FEATURES).isdisjoint({"TX_FRAUD", "TX_FRAUD_SCENARIO"})

model = fit_ranker(train)
print(f"Fitted on {len(train):,} April–July transactions.")
print(f"August validation contains {len(validation):,} transactions.")

## 2 · Compare the August review queues

Each policy sorts eligible transactions by its own score within each day. Ties go to the lower transaction ID. Both policies use up to 50 or 100 review slots per day.

The restricted comparison removes transactions above 220 before ranking, because the simulator labels all of them as fraud. Both policies refill their daily queues from the remaining transactions. Recall uses the fraud count of the selected cohort as its denominator.

In [ ]:
scored = score_transactions(model, validation)
pd.testing.assert_series_equal(scored["TRANSACTION_ID"], validation["TRANSACTION_ID"])
queues, average_precision = compare_review_queues(scored)

assert (queues["Reviews"] == queues["Fraud caught"] + queues["Legitimate reviews"]).all()
assert queues.groupby(["Cohort", "Daily limit"])["Reviews"].nunique().eq(1).all()

queue_display = queues.assign(
    Precision=queues["Precision"].map("{:.1%}".format),
    Recall=queues["Recall"].map("{:.1%}".format),
    **{"Fraud-labelled value caught": queues["Fraud-labelled value caught"].map("{:,.2f}".format)},
)
display(queue_display.set_index(["Cohort", "Daily limit", "Policy"]))
display(average_precision.assign(
    **{"Average precision": average_precision["Average precision"].map("{:.3f}".format)},
).set_index(["Cohort", "Policy"]))

## 3 · Read the difference at equal capacity

The chart shows the change in fraud caught when the model fills the same number of review slots. A positive number means the model placed more labelled fraud into the queue. At equal capacity, every additional fraud case in the queue displaces one legitimate review. The monetary measure is the sum of fraud-labelled transaction amounts in the queue.

In [ ]:
comparison = queues.pivot(index=["Cohort", "Daily limit"], columns="Policy", values="Fraud caught")
change = (comparison["Logistic regression"] - comparison["Amount baseline"]).unstack("Cohort")
figure, axis = plt.subplots(figsize=(8.2, 3.6), layout="constrained")
change.plot.bar(ax=axis, color=["#137A73", "#5B6B89"], rot=0)
axis.axhline(0, color="#334155", linewidth=0.8)
axis.set(title="August validation: change in fraud caught", ylabel="Model minus amount baseline")
axis.set_xlabel("Daily review limit")
axis.legend(frameon=False, title=None)
plt.show()

## Next checkpoint

The [frozen protocol](../docs/methodology.md#frozen-model-and-review-protocol) specifies how to reproduce this model and compare it with the baseline on September. September results will be recorded during P4.